# Improved baseline: four rating bands

Fit bigram TF-IDF and square-root-weighted Spark logistic regression. The archived test output reports 58.50% accuracy and 47.33% macro-F1.

Run from the repository root.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, time, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

PROJECT_DIR = Path.cwd()
DATA_DIR = Path.cwd() / 'data_64_16_20'

SEED = 42
N_CLASSES = 4
CLASS_NAMES = ['low', 'moderate', 'good', 'excellent']
REG_PARAM = 0.1
VOCAB_SIZE = 30000
MIN_DF = 2.0
MAX_ITER = 50  # Cap iterations to limit memory use.

RUN_DIR = PROJECT_DIR / "baseline_results" / datetime.now().strftime("rating_4class_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ================= Configure Spark memory and network timeout =================
spark = (SparkSession.builder.appName("Rating_4class")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.driver.memory", "4g")       # Allocate driver memory.
    .config("spark.executor.memory", "4g")     # Allocate executor memory.
    .config("spark.network.timeout", "600s")   # Allow a longer network timeout.
    .getOrCreate())
# ======================================================================

spark.sparkContext.setLogLevel("ERROR")
print("Output directory:", RUN_DIR)
print("Python:", platform.python_version(), "Spark:", pyspark.__version__)

In [ ]:
# The JSONL input must contain review, rating, and filtered_words.
schema = StructType([
    StructField("review", StringType()),
    StructField("rating", IntegerType()),
    StructField("filtered_words", ArrayType(StringType()))
])

def load_jsonl(path, split_name):
    return spark.read.schema(schema).json(str(path)).withColumn("split", F.lit(split_name))

print("Reading JSONL data...")
df_train = load_jsonl(DATA_DIR / "train.jsonl", "development")
df_val = load_jsonl(DATA_DIR / "validation.jsonl", "development")
df_test = load_jsonl(DATA_DIR / "test.jsonl", "test")

cleanData = df_train.union(df_val).union(df_test)

# Filter invalid records.
valid = F.coalesce(
    F.col("review").isNotNull() & (F.length("review") > 0)
    & F.col("rating").between(1, 10) & (F.size("filtered_words") > 0),
    F.lit(False))

print(f"Rows before filtering: {cleanData.count()}")
cleanData = cleanData.filter(valid)
print(f"Valid rows after filtering: {cleanData.count()}")

cleanData = (cleanData.withColumn("review_original", F.col("review"))
    .withColumn("uniqueID", F.sha2(F.concat_ws("|", F.col("review"), F.col("rating").cast("string")), 256))
    .withColumn("text_group_id", F.sha2(F.to_json("filtered_words"), 256)))

# ===== Map ratings to four classes =====
model_data = cleanData.withColumn("label", 
    F.when(F.col("rating") <= 3, F.lit(0.0))
    .when(F.col("rating") <= 6, F.lit(1.0))
    .when(F.col("rating") <= 8, F.lit(2.0))
    .otherwise(F.lit(3.0))
).cache()

development = model_data.filter(F.col("split") == "development").cache()
test = model_data.filter(F.col("split") == "test").cache()

dev_count = development.count()
test_count = test.count()
print(f"Development rows: {dev_count}, Test rows: {test_count}")

for name, df in [("development", development), ("test", test)]:
    labels = {int(r.label) for r in df.select("label").distinct().collect()}
    assert labels == set(range(4)), f"{name} is missing classes: {labels}"

manifest = model_data.select("text_group_id", "rating", "label", "split").toPandas()
manifest.to_csv(RUN_DIR / "split_manifest.csv", index=False)

distribution = model_data.groupBy("split", "rating").count().orderBy("split", "rating").toPandas()
distribution.to_csv(RUN_DIR / "split_distribution.csv", index=False)
display(distribution.pivot(index="rating", columns="split", values="count"))

In [ ]:
import time
from pyspark.sql.functions import col, when, lit
from pyspark.ml.feature import CountVectorizer, IDF, NGram  # Import the bigram transformer.
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

print("Fitting bigram TF-IDF ...")

# ================= Generate bigram features with NGram =================
# Generate adjacent word pairs and pass them to CountVectorizer as tokens.
ngram = NGram(n=2, inputCol="filtered_words", outputCol="ngrams")

# Add a bigram column to the development and test data.
ngram_development = ngram.transform(development)
ngram_test = ngram.transform(test)

# Vectorize the bigrams and apply IDF weighting.
feature_model = Pipeline(stages=[
    CountVectorizer(
        inputCol="ngrams",       # Use the generated "ngrams" column.
        outputCol="term_counts", 
        vocabSize=VOCAB_SIZE, 
        minDF=MIN_DF, 
        binary=False
    ),
    IDF(inputCol="term_counts", outputCol="features")
]).fit(ngram_development)
# ==================================================================

development_features = feature_model.transform(ngram_development).cache()
test_features = feature_model.transform(ngram_test).cache()

# ================= Apply square-root class weighting =================
print("Computing square-root class weights ...")
total_count = development_features.count()
counts_dict = {int(row['label']): row['count'] for row in development_features.groupBy("label").count().collect()}

weight_expr = None
for label_val, count in counts_dict.items():
    # Weight each class by the square root of total samples divided by its sample count.
    w = (float(total_count) / count) ** 0.5
    print(f"Label {label_val}: count={count}, weight={w:.4f}")
    if weight_expr is None:
        weight_expr = when(col("label") == label_val, lit(w))
    else:
        weight_expr = weight_expr.when(col("label") == label_val, lit(w))

dev_weighted = development_features.withColumn("weight", weight_expr).cache()
# =============================================================

print("Training logistic regression with bigrams and square-root class weights ...")
start = time.perf_counter()
lr = LogisticRegression(
    featuresCol="features", 
    labelCol="label", 
    weightCol="weight",  # Apply the class weights during training.
    family="multinomial", 
    regParam=REG_PARAM,
    elasticNetParam=0.0, 
    maxIter=MAX_ITER,
    standardization=True, 
    tol=1e-6
)

model = lr.fit(dev_weighted) 
iterations = int(model.summary.totalIterations) if hasattr(model.summary, 'totalIterations') else MAX_ITER

print(f"Training completed in  {time.perf_counter() - start:.2f}  seconds, Iterations: {iterations}, Vocabulary size: {len(feature_model.stages[0].vocabulary)}")

# Save model parameters.
(RUN_DIR / "feature_metadata.json").write_text(json.dumps({"vocabulary": feature_model.stages[0].vocabulary}, ensure_ascii=False), encoding="utf-8")
np.savez_compressed(RUN_DIR / "parameters.npz", idf=feature_model.stages[1].idf.toArray(), coefficients=model.coefficientMatrix.toArray(), intercept=model.interceptVector.toArray())
pd.DataFrame({"iteration": range(len(model.summary.objectiveHistory)), "objective": model.summary.objectiveHistory}).to_csv(RUN_DIR / "objective_history.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import json
import matplotlib.pyplot as plt
from pyspark.ml.functions import vector_to_array

# Map each rating band to a representative score for MAE.
RATING_CENTERS = [2.0, 5.0, 7.5, 9.5] 

def evaluate_4class_model(predictions):
    local = predictions.select("uniqueID", "rating", vector_to_array("probability").alias("probability")).toPandas()
    y_true_rating = local.rating.to_numpy(dtype=int)
    prob_matrix = np.stack(local.probability.to_numpy())
    
    pred_class = prob_matrix.argmax(axis=1)
    assert pred_class.max() <= 3, f"Predicted class {pred_class.max()} is out of range. Check the training labels."
    
    pred_rating = np.array([RATING_CENTERS[c] for c in pred_class])
    
    # Map the true class to its representative rating.
    y_true_class = np.where(y_true_rating <= 3, 0, 
                   np.where(y_true_rating <= 6, 1, 
                   np.where(y_true_rating <= 8, 2, 3)))
    
    report = classification_report(
        y_true_class, pred_class, 
        labels=range(4), 
        target_names=CLASS_NAMES, 
        output_dict=True, 
        zero_division=0
    )
    
    mae = float(np.mean(np.abs(pred_rating - y_true_rating)))
    rmse = float(np.sqrt(np.mean((pred_rating - y_true_rating) ** 2)))
    within_1 = float(np.mean(np.abs(pred_rating - y_true_rating) <= 1.5))
    within_2 = float(np.mean(np.abs(pred_rating - y_true_rating) <= 2.5))
    
    summary = {
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"],
        "mae_expected": mae,
        "rmse_expected": rmse,
        "within_1": within_1,
        "within_2": within_2
    }
    
    cm = confusion_matrix(y_true_class, pred_class, labels=range(4))
    
    local["pred_class"] = pred_class
    local["pred_rating"] = pred_rating
    local["true_class"] = y_true_class
    local["probability"] = local.probability.map(lambda x: json.dumps([float(v) for v in x]))
    
    return summary, pd.DataFrame(report).transpose(), cm, local

print("Evaluating on the test set ...")
predictions = model.transform(test_features)
summary, report_df, cm, local_predictions = evaluate_4class_model(predictions)

(RUN_DIR / "final_test_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
report_df.to_csv(RUN_DIR / "final_test_classification_report.csv")
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RUN_DIR / "final_test_confusion_matrix.csv")
local_predictions.to_csv(RUN_DIR / "final_test_predictions.csv", index=False)

print("\n" + "="*20 + " Final test results (weighted four-class model) " + "="*20)
print(f"accuracy: {summary['accuracy']:.4f}")
print(f"macro_f1: {summary['macro_f1']:.4f}")
print(f"precision_macro: {summary['precision_macro']:.4f}")
print(f"recall_macro: {summary['recall_macro']:.4f}")
print("-" * 50)
print(f"MAE for mapped expected ratings: {summary['mae_expected']:.4f} | Accuracy within 1.5 points: {summary['within_1']:.4f}")
print("="*50 + "\n")

display(report_df)

# Plot results.
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       xlabel="Predicted Category", ylabel="Observed Category", title="4-Class Confusion Matrix (Weighted)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", 
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(RUN_DIR / "test_confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)